# Privacy Filter BR v3 — Hybrid Taxonomy Fine-tune

Base: `openai/privacy-filter` (1.5B / 50M active MoE)

**Strategy:** Reuse original head (8 categories, 33 labels) + extend with 14 BR categories = 22 total / 89 labels. Original head weights are copied; only new positions are randomly initialized.

**Categories retained from original:** `account_number, private_address, private_date, private_email, private_person, private_phone, private_url, secret`

**Categories added (BR):** `private_cpf, private_cnpj, private_rg, private_cnh, private_pis, private_titulo_eleitor, private_certidao, private_ie, private_order_id, private_tracking_code, private_invoice_number, private_client_revenue, private_transaction_id, private_customer_id`

**Anti-forgetting:** Dataset includes ~2k synthetic examples covering the OAI-only categories (date/url/secret/account_number) so the model doesn't lose those capabilities during fine-tune.

Runtime: A100, ~45-75min for 3 epochs on ~54k examples.

## Files to upload to /content/
- `dataset_br_v3.jsonl`
- `dataset_br_v3_holdout.jsonl`
- `finetune_v3.py`

In [ ]:
!pip install -q transformers peft datasets seqeval accelerate

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

In [ ]:
!ls -la /content/dataset_br_v3*.jsonl /content/finetune_v3.py
!wc -l /content/dataset_br_v3*.jsonl

In [ ]:
!cd /content && python finetune_v3.py \
    --train-file /content/dataset_br_v3.jsonl \
    --eval-file /content/dataset_br_v3_holdout.jsonl \
    --output-dir /content/privacy-filter-br-v3 \
    --epochs 3 \
    --batch-size 16 \
    --lr 2e-4

In [ ]:
!cat /content/privacy-filter-br-v3/benchmark.txt

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch

MODEL_DIR = '/content/privacy-filter-br-v3'
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR, dtype=torch.float32).cuda().eval()

ner = pipeline('token-classification', model=model, tokenizer=tok, aggregation_strategy='simple', device=0)

samples = [
    # BR PII
    'João Silva, CPF 123.456.789-00, comprou no pedido ML-2024-789456.',
    'Empresa Acme Ltda, CNPJ 12.345.678/0001-90, faturou R$ 50.000,00.',
    # OAI retained
    'Acesse https://meusaas.com/reset?token=abc123def456 antes de 15/01/2026.',
    'API Key: sk-proj-abc1234567890def. Cartão 4111 1111 1111 1111.',
]
for s in samples:
    print('TEXT:', s)
    for ent in ner(s):
        print(f'  {ent["entity_group"]:25} | {ent["word"]!r} | score={ent["score"]:.3f}')
    print()

In [ ]:
!cd /content && zip -r privacy-filter-br-v3.zip privacy-filter-br-v3/
!ls -lh /content/privacy-filter-br-v3.zip

from google.colab import files
files.download('/content/privacy-filter-br-v3.zip')